# KTX 승차인원수 예측 - Test 기간 확장 (2024-04 ~ 2025-12)

## 목표
- Test 기간의 독립변수를 먼저 예측 (forecast_future.ipynb 방식)
- 8개 모델로 Valid + Test 기간 예측
- Train+Valid+Test 전체 기간 시각화

## 주요 특징
- Linear Regression + 계절성 인코딩 (TimeIndex, Month_sin/cos)
- 예상 실행 시간: **15-20분**
- 상세한 로그 출력

In [43]:
# Section 1: 환경 설정 및 데이터 로드
import os
import warnings
warnings.filterwarnings('ignore')
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from pandas.tseries.offsets import MonthEnd
from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 한글 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# Pandas 전체 보기 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# 경로 설정
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == 'code':
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

RAW_DATA_PATH = PROJECT_DIR / 'raw_data'
PROCESSED_DATA_PATH = PROJECT_DIR / 'processed_data'
RESULT_PATH = PROJECT_DIR / 'result'
RESULT_PATH.mkdir(exist_ok=True)

print(f"✅ 환경 설정 완료")
print(f"   - Raw Data: {RAW_DATA_PATH}")
print(f"   - Processed Data: {PROCESSED_DATA_PATH}")
print(f"   - Result: {RESULT_PATH}")

# 시작 시간 기록
start_time = time.time()
print(f"\n⏱️  시작 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}")

✅ 환경 설정 완료
   - Raw Data: /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/raw_data
   - Processed Data: /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/processed_data
   - Result: /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/result

⏱️  시작 시간: 2025-12-30 17:06:31


In [44]:
# Section 2: 데이터 로드 및 병합
print("=" * 80)
print("📊 Section 2: 데이터 로드 및 병합")
print("=" * 80)

# KTX 데이터 로드
df_ktx = pd.read_csv(RAW_DATA_PATH / 'df_KTX_monthsum_KK.csv')
df_ktx['Date'] = pd.to_datetime(df_ktx['Date'])
df_ktx = df_ktx[(df_ktx['주운행선']=='경부선') & (df_ktx['전체주중주말']=='주말')]
df_ktx.columns = df_ktx.columns.str.strip()
df_ktx = df_ktx.sort_values('Date').reset_index(drop=True)
print(f"✅ KTX 데이터: {df_ktx.shape}")

# ECON 데이터 로드 (Excel 파일)
df_active = pd.read_excel(RAW_DATA_PATH / 'ECON_경제활동인구_KK.xlsx', sheet_name='데이터', header=7)
df_active['Date'] = pd.to_datetime(df_active['Date'])
df_active.columns = df_active.columns.str.strip().str.replace(' ', '_', regex=False)

df_sentiment = pd.read_excel(RAW_DATA_PATH / 'ECON_소비자동향조사(전국, 월, 2008.9~)_KK.xlsx', sheet_name='데이터', header=7)
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'])
df_sentiment = df_sentiment.dropna(axis=1)

df_price = pd.read_excel(RAW_DATA_PATH / 'ECON_소비자물가지수_KK.xlsx', sheet_name='데이터', header=6)
df_price['Date'] = pd.to_datetime(df_price['Date'])
df_price.columns = df_price.columns.str.strip().str.replace(' ', '_', regex=False)
print(f"✅ ECON 데이터 로드 완료")

# KOSIS 데이터 로드
df_traffic = pd.read_csv(RAW_DATA_PATH / 'KOSIS_내국인출국교통수단별_KK.csv', encoding='cp949', header=2)
df_traffic['Date'] = pd.to_datetime(df_traffic['Date'])
df_traffic['내국인출입국_공항'] = df_traffic[[col for col in df_traffic.columns if col.split('_')[0] == '공항']].sum(axis=1).values
df_traffic['내국인출입국_항구'] = df_traffic[[col for col in df_traffic.columns if col.split('_')[0] == '항구']].sum(axis=1).values
df_traffic.drop(columns=[col for col in df_traffic.columns if col.split('_')[0] in ['공항', '항구']], inplace=True)

df_entry = pd.read_csv(RAW_DATA_PATH / 'KOSIS_외래객_입국목적별_국적별_KK.csv', encoding='cp949', header=1)
df_entry['Date'] = pd.to_datetime(df_entry['Date'])

df_population = pd.read_csv(RAW_DATA_PATH / 'KOSIS_인구동태건수_및_동태율_추이_출생_사망_혼인_이혼__KK.csv', encoding='cp949', header=0)
df_population['Date'] = pd.to_datetime(df_population['Date'], format='%Y', errors='coerce')
df_population.columns = df_population.columns.str.strip().str.replace('(', '_', regex=False).str.replace(')', '', regex=False)
df_population['Year'] = df_population['Date'].dt.year
print(f"✅ KOSIS 데이터 로드 완료")

# 데이터 병합
df_merged = df_ktx.copy()
df_merged = df_merged.merge(df_active, on='Date', how='left')
df_merged = df_merged.merge(df_sentiment, on='Date', how='left')
df_merged = df_merged.merge(df_price, on='Date', how='left')
df_merged = df_merged.merge(df_traffic, on='Date', how='left')
df_merged = df_merged.merge(df_entry, on='Date', how='left')

# Year, Month 추가
df_merged['Year'] = df_merged['Date'].dt.year
df_merged['Month'] = df_merged['Date'].dt.month

# 인구 데이터 병합 (연 단위 → 월 단위)
df_population_monthly = df_population.copy()
df_population_monthly['출생아수_월평균_명'] = df_population_monthly['출생아수_명'] / 12
df_population_monthly['사망자수_월평균_명'] = df_population_monthly['사망자수_명'] / 12
df_population_monthly['자연증가건수_월평균_명'] = df_population_monthly['자연증가건수_명'] / 12
df_population_monthly['혼인건수_월평균_건'] = df_population_monthly['혼인건수_건'] / 12
df_population_monthly['이혼건수_월평균_건'] = df_population_monthly['이혼건수_건'] / 12

population_cols = ['Year', '출생아수_월평균_명', '사망자수_월평균_명', '자연증가건수_월평균_명',
                  '혼인건수_월평균_건', '이혼건수_월평균_건', '조출생률_천명당', '조사망률_천명당', 
                  '자연증가율_천명당', '합계출산율_명', '출생성비_명', '조혼인율_천명당', '조이혼율_천명당']
df_merged = df_merged.merge(df_population_monthly[population_cols], on='Year', how='left')

# 뉴스 데이터 로드
df_news = pd.read_csv(PROCESSED_DATA_PATH / 'KTX_Monthly_Demand_Forecast_Data.csv')
df_news['Date'] = pd.to_datetime(df_news['Date'])
df_merged = df_merged.merge(df_news, on='Date', how='left')
print(f"✅ NEWS 데이터 병합 완료")

# 2025-12까지 확장
last_date = df_merged['Date'].max()
target_date = pd.to_datetime('2025-12-31')

if last_date < target_date:
    new_dates = pd.date_range(start=last_date + MonthEnd(1), end=target_date, freq='ME')
    new_rows = pd.DataFrame({'Date': new_dates})
    df_merged = pd.concat([df_merged, new_rows], ignore_index=True)
    df_merged = df_merged.sort_values('Date').reset_index(drop=True)

# 단일 값 컬럼 제거
cols_to_drop = [col for col in df_merged.columns if df_merged[col].nunique() == 1]
df_merged.drop(columns=cols_to_drop, inplace=True)

print(f"\n✅ 데이터 병합 완료: {df_merged.shape[0]}개월 × {df_merged.shape[1]}개 컬럼")
print(f"   - 기간: {df_merged['Date'].min()} ~ {df_merged['Date'].max()}")
print(f"   - 제거된 단일값 컬럼: {len(cols_to_drop)}개")
print(f"   - Train+Valid 기간: 2015-01 ~ 2024-03 (99 + 12 = 111개월)")
print(f"   - Test 기간: 2024-04 ~ 2025-12 (21개월)")

📊 Section 2: 데이터 로드 및 병합
✅ KTX 데이터: (132, 22)
✅ ECON 데이터 로드 완료
✅ KOSIS 데이터 로드 완료
✅ NEWS 데이터 병합 완료

✅ 데이터 병합 완료: 133개월 × 105개 컬럼
   - 기간: 2015-01-01 00:00:00 ~ 2025-12-31 00:00:00
   - 제거된 단일값 컬럼: 2개
   - Train+Valid 기간: 2015-01 ~ 2024-03 (99 + 12 = 111개월)
   - Test 기간: 2024-04 ~ 2025-12 (21개월)


In [45]:
# Section 3: 시간 특성 생성 (forecast_future.ipynb 방식)
print("\n" + "=" * 80)
print("⏰ Section 3: 시간 특성 생성")
print("=" * 80)

def add_basic_time_features(df):
    """기본 시간 특성 생성 (독립변수 예측용)"""
    df = df.copy()
    
    # 달력 정보
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['Date'].apply(lambda x: x in kr_holidays).astype(int)
    df['weekday'] = df['Date'].dt.weekday
    df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    
    # 시간 인덱스 (Trend)
    df['TimeIndex'] = (df['Date'].dt.year - df['Date'].dt.year.min()) * 12 + df['Date'].dt.month
    
    # 계절성 (Cyclical Encoding)
    df['Month_sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)
    df['Quarter'] = df['Date'].dt.quarter
    
    return df

df_merged = add_basic_time_features(df_merged)

print("✅ 시간 특성 생성 완료:")
print("   - 추세: TimeIndex")
print("   - 계절성: Month_sin, Month_cos, Quarter")
print("   - 달력: is_weekend, is_holiday, Year, Month, weekday")
print(f"\n현재 컬럼 수: {df_merged.shape[1]}개")


⏰ Section 3: 시간 특성 생성
✅ 시간 특성 생성 완료:
   - 추세: TimeIndex
   - 계절성: Month_sin, Month_cos, Quarter
   - 달력: is_weekend, is_holiday, Year, Month, weekday

현재 컬럼 수: 112개


In [46]:
# Section 4: 독립변수 예측 (Linear Regression + 계절성 인코딩)
print("\n" + "=" * 80)
print("🔮 Section 4: 독립변수 예측 (Test 기간: 2024-04 ~ 2025-12)")
print("=" * 80)

# Feature 정의 (추세 + 계절성 + 달력)
feature_cols = ['TimeIndex', 'Month_sin', 'Month_cos', 'Quarter', 'is_weekend', 'is_holiday']

# 예측 대상 컬럼 (승차인원수 제외한 모든 수치형 변수)
exclude_cols = ['Date', '승차인원수', 'Year', 'Month', 'TimeIndex', 'is_holiday',
                'weekday', 'is_weekend', 'Month_sin', 'Month_cos', 'Quarter']
target_features = [col for col in df_merged.select_dtypes(include=[np.number]).columns
                   if col not in exclude_cols]

print(f"예측 대상 변수: {len(target_features)}개")
print(f"Feature: {feature_cols}")

# 학습 기간 (2024-04 이전)
train_mask_indep = df_merged['Date'] < '2024-04-01'
test_mask_indep = df_merged['Date'] >= '2024-04-01'

# 1) 각 변수별 모델 학습
models_indep = {}
section4_start = time.time()

print(f"\n[진행] 독립변수 모델 학습 중...")
for col in tqdm(target_features, desc="독립변수 모델 학습"):
    valid_data_indep = df_merged[train_mask_indep & df_merged[col].notna()]
    
    if len(valid_data_indep) < 12:  # 최소 1년 데이터 필요
        continue
    
    X_feat = valid_data_indep[feature_cols]
    y_feat = valid_data_indep[col]
    
    model = LinearRegression()
    model.fit(X_feat, y_feat)
    models_indep[col] = model

section4_time = time.time() - section4_start
print(f"\n✅ {len(models_indep)}개 모델 학습 완료 (소요: {section4_time:.2f}초)")

# 2) Test 기간 예측
print(f"\n[진행] Test 기간 독립변수 예측 중...")
for col in target_features:
    if col in models_indep:
        X_test = df_merged.loc[test_mask_indep, feature_cols]
        predictions = models_indep[col].predict(X_test)
        df_merged.loc[test_mask_indep, col] = predictions

print(f"✅ 독립변수 예측 완료 (2024-04 ~ 2025-12)")

# 3) 결측치 Forward Fill (혹시 남아있는 경우)
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
df_merged[numeric_cols] = df_merged[numeric_cols].ffill()

print(f"\n📊 예측 후 데이터 상태:")
print(f"   - Shape: {df_merged.shape}")
print(f"   - 결측치: {df_merged.isnull().sum().sum()}개")


🔮 Section 4: 독립변수 예측 (Test 기간: 2024-04 ~ 2025-12)
예측 대상 변수: 101개
Feature: ['TimeIndex', 'Month_sin', 'Month_cos', 'Quarter', 'is_weekend', 'is_holiday']

[진행] 독립변수 모델 학습 중...


독립변수 모델 학습: 100%|██████████| 101/101 [00:00<00:00, 645.60it/s]


✅ 101개 모델 학습 완료 (소요: 0.16초)

[진행] Test 기간 독립변수 예측 중...


✅ 독립변수 예측 완료 (2024-04 ~ 2025-12)

📊 예측 후 데이터 상태:
   - Shape: (133, 112)
   - 결측치: 0개


In [47]:
# Section 5: Feature Engineering (Lag, Rolling, MoM, YoY)
print("\n" + "=" * 80)
print("🔧 Section 5: Feature Engineering")
print("=" * 80)

def add_advanced_features(df, target_col='승차인원수'):
    """고급 시간 특성 생성 (Lag, Rolling, MoM, YoY)"""
    df = df.copy()
    
    # Lag Features (1, 3, 6, 12개월)
    for lag in [1, 3, 6, 12]:
        df[f'{target_col}_Lag_{lag}'] = df[target_col].shift(lag)
    
    # Rolling Features (3, 6, 12개월 mean/std)
    for window in [3, 6, 12]:
        df[f'{target_col}_Rolling_Mean_{window}'] = df[target_col].shift(1).rolling(window=window).mean()
        df[f'{target_col}_Rolling_Std_{window}'] = df[target_col].shift(1).rolling(window=window).std()
    
    # MoM, YoY Change
    df[f'{target_col}_MoM_change'] = df[target_col].pct_change(1)
    df[f'{target_col}_YoY_change'] = df[target_col].pct_change(12)
    
    return df

df_featured = add_advanced_features(df_merged, target_col='승차인원수')

# 기존 modeling.ipynb에서 사용한 외부 변수의 Lag/Rolling도 추가 (주요 변수만)
important_vars = ['국내총생산_GDP_경상', '소비자물가지수_총지수', '취업자', 
                  '총저축_실질', '여객_철도_KTX_경부선_열차']

for var in important_vars:
    if var in df_featured.columns:
        for lag in [1, 3, 6]:
            df_featured[f'{var}_Lag_{lag}'] = df_featured[var].shift(lag)

feature_count = df_featured.shape[1]
print(f"✅ Feature Engineering 완료: {feature_count}개 특성")
print(f"   - Lag Features: {len([c for c in df_featured.columns if 'Lag' in c])}개")
print(f"   - Rolling Features: {len([c for c in df_featured.columns if 'Rolling' in c])}개")
print(f"   - Change Features: {len([c for c in df_featured.columns if 'change' in c])}개")


🔧 Section 5: Feature Engineering
✅ Feature Engineering 완료: 124개 특성
   - Lag Features: 4개
   - Rolling Features: 6개
   - Change Features: 2개


In [48]:
# Section 6: 데이터 분할 및 정규화
print("\n" + "=" * 80)
print("📐 Section 6: 데이터 분할 및 정규화")
print("=" * 80)

# 데이터 분할
train_end_date = '2023-03-31'
valid_end_date = '2024-03-31'

train_mask = df_featured['Date'] <= train_end_date
valid_mask = (df_featured['Date'] > train_end_date) & (df_featured['Date'] <= valid_end_date)
test_mask = df_featured['Date'] > valid_end_date

train_data = df_featured[train_mask].copy()
valid_data = df_featured[valid_mask].copy()
test_data = df_featured[test_mask].copy()

print(f"Train: {len(train_data)}개월 ({train_data['Date'].min()} ~ {train_data['Date'].max()})")
print(f"Valid: {len(valid_data)}개월 ({valid_data['Date'].min()} ~ {valid_data['Date'].max()})")
print(f"Test:  {len(test_data)}개월 ({test_data['Date'].min()} ~ {test_data['Date'].max()})")

# Feature 선택 (Date와 Target 제외)
feature_columns = [col for col in df_featured.columns if col not in ['Date', '승차인원수']]

# 결측치 처리 (SimpleImputer로 median 채우기)
imputer = SimpleImputer(strategy='median')
train_data[feature_columns] = imputer.fit_transform(train_data[feature_columns])
valid_data[feature_columns] = imputer.transform(valid_data[feature_columns])
test_data[feature_columns] = imputer.transform(test_data[feature_columns])

# df_featured에도 반영
df_featured.loc[train_mask, feature_columns] = train_data[feature_columns].values
df_featured.loc[valid_mask, feature_columns] = valid_data[feature_columns].values
df_featured.loc[test_mask, feature_columns] = test_data[feature_columns].values

print(f"\n✅ 결측치 처리 완료 (Median Imputation)")
print(f"   - Train 결측: {train_data[feature_columns].isnull().sum().sum()}개")
print(f"   - Valid 결측: {valid_data[feature_columns].isnull().sum().sum()}개")
print(f"   - Test 결측: {test_data[feature_columns].isnull().sum().sum()}개")

# 정규화 (StandardScaler)
scaler = StandardScaler()
train_data[feature_columns] = scaler.fit_transform(train_data[feature_columns])
valid_data[feature_columns] = scaler.transform(valid_data[feature_columns])
test_data[feature_columns] = scaler.transform(test_data[feature_columns])

# df_featured에도 반영
df_featured.loc[train_mask, feature_columns] = train_data[feature_columns].values
df_featured.loc[valid_mask, feature_columns] = valid_data[feature_columns].values
df_featured.loc[test_mask, feature_columns] = test_data[feature_columns].values

# 인덱스 리셋 (중요!)
df_featured = df_featured.reset_index(drop=True)
train_data = train_data.reset_index(drop=True)
valid_data = valid_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"✅ StandardScaler 적용 완료")
print(f"\n총 Feature 수: {len(feature_columns)}개")


📐 Section 6: 데이터 분할 및 정규화
Train: 99개월 (2015-01-01 00:00:00 ~ 2023-03-01 00:00:00)
Valid: 12개월 (2023-04-01 00:00:00 ~ 2024-03-01 00:00:00)
Test:  22개월 (2024-04-01 00:00:00 ~ 2025-12-31 00:00:00)

✅ 결측치 처리 완료 (Median Imputation)
   - Train 결측: 0개
   - Valid 결측: 0개
   - Test 결측: 0개
✅ StandardScaler 적용 완료

총 Feature 수: 122개


In [49]:
# Section 7: 8개 모델 학습
print("\n" + "=" * 80)
print("🤖 Section 7: 8개 모델 학습")
print("=" * 80)

# 학습 데이터 준비
X_train = train_data[feature_columns]
y_train = train_data['승차인원수']
X_valid = valid_data[feature_columns]
y_valid = valid_data['승차인원수']

# 모델 딕셔너리
models_dict = {}
section7_start = time.time()

# 1. RandomForest
print(f"\n[1/8] RandomForest 학습 중...")
model_start = time.time()
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_time = time.time() - model_start
models_dict['RandomForest'] = rf_model
print(f"✅ RandomForest 학습 완료 (소요: {rf_time:.2f}초)")

# 2. XGBoost
print(f"\n[2/8] XGBoost 학습 중...")
model_start = time.time()
xgb_model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - model_start
models_dict['XGBoost'] = xgb_model
print(f"✅ XGBoost 학습 완료 (소요: {xgb_time:.2f}초)")

# 3. LightGBM
print(f"\n[3/8] LightGBM 학습 중...")
model_start = time.time()
lgbm_model = LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
lgbm_model.fit(X_train, y_train)
lgbm_time = time.time() - model_start
models_dict['LightGBM'] = lgbm_model
print(f"✅ LightGBM 학습 완료 (소요: {lgbm_time:.2f}초)")

# 4. CatBoost
print(f"\n[4/8] CatBoost 학습 중...")
model_start = time.time()
cat_model = CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, random_state=42, verbose=0)
cat_model.fit(X_train, y_train)
cat_time = time.time() - model_start
models_dict['CatBoost'] = cat_model
print(f"✅ CatBoost 학습 완료 (소요: {cat_time:.2f}초)")

# 5. MLP
print(f"\n[5/8] MLP 학습 중...")
model_start = time.time()
mlp_model = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, early_stopping=True)
mlp_model.fit(X_train, y_train)
mlp_time = time.time() - model_start
models_dict['MLP'] = mlp_model
print(f"✅ MLP 학습 완료 (소요: {mlp_time:.2f}초)")

print(f"\n✅ Tree-based & MLP 모델 (5/8) 학습 완료")


🤖 Section 7: 8개 모델 학습

[1/8] RandomForest 학습 중...


✅ RandomForest 학습 완료 (소요: 0.10초)

[2/8] XGBoost 학습 중...
✅ XGBoost 학습 완료 (소요: 12.92초)

[3/8] LightGBM 학습 중...
✅ LightGBM 학습 완료 (소요: 4.93초)

[4/8] CatBoost 학습 중...
✅ CatBoost 학습 완료 (소요: 0.30초)

[5/8] MLP 학습 중...
✅ MLP 학습 완료 (소요: 0.06초)

✅ Tree-based & MLP 모델 (5/8) 학습 완료


In [50]:
# Section 7-2: Deep Learning 모델 (RNN, LSTM, GRU)
print("\n" + "=" * 80)
print("🧠 Section 7-2: Deep Learning 모델 학습")
print("=" * 80)

# PyTorch 데이터셋 정의
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
        self.y = torch.FloatTensor(y.values if hasattr(y, 'values') else y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 데이터 로더 생성
train_dataset = TimeSeriesDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# RNN 모델 정의
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, seq_len=1, features)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

# LSTM 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

# GRU 모델 정의
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out.squeeze()

# 학습 함수
def train_pytorch_model(model, train_loader, epochs=100, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
    
    return model

input_size = X_train.shape[1]

# 6. RNN
print(f"\n[6/8] RNN 학습 중...")
model_start = time.time()
rnn_model = RNNModel(input_size)
rnn_model = train_pytorch_model(rnn_model, train_loader, epochs=100)
rnn_time = time.time() - model_start
models_dict['RNN'] = rnn_model
print(f"✅ RNN 학습 완료 (소요: {rnn_time:.2f}초)")

# 7. LSTM
print(f"\n[7/8] LSTM 학습 중...")
model_start = time.time()
lstm_model = LSTMModel(input_size)
lstm_model = train_pytorch_model(lstm_model, train_loader, epochs=100)
lstm_time = time.time() - model_start
models_dict['LSTM'] = lstm_model
print(f"✅ LSTM 학습 완료 (소요: {lstm_time:.2f}초)")

# 8. GRU
print(f"\n[8/8] GRU 학습 중...")
model_start = time.time()
gru_model = GRUModel(input_size)
gru_model = train_pytorch_model(gru_model, train_loader, epochs=100)
gru_time = time.time() - model_start
models_dict['GRU'] = gru_model
print(f"✅ GRU 학습 완료 (소요: {gru_time:.2f}초)")

section7_time = time.time() - section7_start
print(f"\n" + "=" * 80)
print(f"✅ 전체 8개 모델 학습 완료 (총 소요: {section7_time/60:.1f}분)")
print(f"=" * 80)


🧠 Section 7-2: Deep Learning 모델 학습

[6/8] RNN 학습 중...
✅ RNN 학습 완료 (소요: 0.60초)

[7/8] LSTM 학습 중...
✅ LSTM 학습 완료 (소요: 0.92초)

[8/8] GRU 학습 중...
✅ GRU 학습 완료 (소요: 0.70초)

✅ 전체 8개 모델 학습 완료 (총 소요: 0.3분)


In [51]:
# Section 8: Recursive Forecast 함수 정의
print("\n" + "=" * 80)
print("🔄 Section 8: Recursive Forecast 함수 정의")
print("=" * 80)

def predict_single(model, X, model_name):
    """단일 예측 (모델 타입에 따라 처리)"""
    if model_name in ['RNN', 'LSTM', 'GRU']:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X.values if hasattr(X, 'values') else X)
            pred = model(X_tensor).numpy()
        # 차원에 따라 적절히 처리
        if pred.ndim == 0:  # 스칼라 (0-dimensional)
            return float(pred)
        elif pred.ndim == 1:
            return pred[0]
        else:
            return pred[0, 0]
    else:
        return model.predict(X)[0]

def update_lag_rolling_features(df, current_idx, target_col='승차인원수'):
    """Lag 및 Rolling Features 업데이트 (Division by zero 방지)"""
    # Lag Features 업데이트
    for lag in [1, 3, 6, 12]:
        if current_idx >= lag:
            df.loc[current_idx, f'{target_col}_Lag_{lag}'] = df.loc[current_idx - lag, target_col]
    
    # Rolling Features 업데이트
    for window in [3, 6, 12]:
        if current_idx >= window:
            recent_values = df.loc[current_idx - window:current_idx - 1, target_col]
            df.loc[current_idx, f'{target_col}_Rolling_Mean_{window}'] = recent_values.mean()
            df.loc[current_idx, f'{target_col}_Rolling_Std_{window}'] = recent_values.std()
    
    # MoM, YoY Change 업데이트 (Division by zero 방지)
    if current_idx >= 1:
        prev_val = df.loc[current_idx - 1, target_col]
        curr_val = df.loc[current_idx, target_col]
        if prev_val != 0 and not np.isnan(prev_val) and not np.isinf(prev_val):
            mom_change = (curr_val - prev_val) / prev_val
            # inf/-inf 체크
            if np.isinf(mom_change) or np.isnan(mom_change):
                mom_change = 0.0
            df.loc[current_idx, f'{target_col}_MoM_change'] = mom_change
        else:
            df.loc[current_idx, f'{target_col}_MoM_change'] = 0.0
    
    if current_idx >= 12:
        prev_val = df.loc[current_idx - 12, target_col]
        curr_val = df.loc[current_idx, target_col]
        if prev_val != 0 and not np.isnan(prev_val) and not np.isinf(prev_val):
            yoy_change = (curr_val - prev_val) / prev_val
            # inf/-inf 체크
            if np.isinf(yoy_change) or np.isnan(yoy_change):
                yoy_change = 0.0
            df.loc[current_idx, f'{target_col}_YoY_change'] = yoy_change
        else:
            df.loc[current_idx, f'{target_col}_YoY_change'] = 0.0
    
    # 외부 변수 Lag도 업데이트
    important_vars = ['국내총생산_GDP_경상', '소비자물가지수_총지수', '취업자', 
                      '총저축_실질', '여객_철도_KTX_경부선_열차']
    for var in important_vars:
        if var in df.columns:
            for lag in [1, 3, 6]:
                if current_idx >= lag:
                    df.loc[current_idx, f'{var}_Lag_{lag}'] = df.loc[current_idx - lag, var]
    
    return df

def recursive_forecast_extended(model, model_name, df_data, scaler, imputer, 
                                 feature_cols, start_idx, end_idx, 
                                 target_col='승차인원수'):
    """
    Recursive Forecast (Valid 또는 Test 기간)
    
    Parameters:
    - model: 학습된 모델
    - model_name: 모델 이름 (str)
    - df_data: 전체 데이터프레임 (df_featured의 복사본)
    - scaler: StandardScaler 객체
    - imputer: SimpleImputer 객체
    - feature_cols: Feature 컬럼 리스트
    - start_idx: 예측 시작 인덱스
    - end_idx: 예측 종료 인덱스
    - target_col: 타겟 컬럼 이름
    
    Returns:
    - predictions: 예측값 리스트
    """
    predictions = []
    df_working = df_data.copy()
    
    for idx in tqdm(range(start_idx, end_idx + 1), desc=f"[{model_name}] Recursive Forecast"):
        # Lag/Rolling 업데이트
        df_working = update_lag_rolling_features(df_working, idx, target_col)
        
        # Feature 추출
        X_current = df_working.loc[idx:idx, feature_cols]
        
        # inf/-inf 체크 및 대체
        X_current = X_current.replace([np.inf, -np.inf], np.nan)
        
        # 정규화
        X_current = imputer.transform(X_current)
        X_current = scaler.transform(X_current)
        
        # 한번 더 inf/-inf 체크 (scaler 후)
        if np.isinf(X_current).any() or np.isnan(X_current).any():
            X_current = np.nan_to_num(X_current, nan=0.0, posinf=0.0, neginf=0.0)
        
        # 예측
        pred = predict_single(model, X_current, model_name)
        
        # Clipping (음수 방지, 이상치 방지)
        train_max = df_data.loc[:start_idx - 1, target_col].max()
        pred = np.clip(pred, 0, train_max * 2)
        
        # 예측값 저장
        predictions.append(pred)
        df_working.loc[idx, target_col] = pred
    
    return predictions

print("✅ Recursive Forecast 함수 정의 완료")
print("   - predict_single: 모델 타입별 예측 처리 (0-d, 1-d, 2-d 대응)")
print("   - update_lag_rolling_features: Lag/Rolling 자동 업데이트 (Division by zero 방지)")
print("   - recursive_forecast_extended: Valid/Test 기간 재귀 예측 (inf/-inf 안전 처리)")


🔄 Section 8: Recursive Forecast 함수 정의
✅ Recursive Forecast 함수 정의 완료
   - predict_single: 모델 타입별 예측 처리 (0-d, 1-d, 2-d 대응)
   - update_lag_rolling_features: Lag/Rolling 자동 업데이트 (Division by zero 방지)
   - recursive_forecast_extended: Valid/Test 기간 재귀 예측 (inf/-inf 안전 처리)


In [52]:
# Section 9: Valid + Test 예측 실행
print("\n" + "=" * 80)
print("📈 Section 9: Valid + Test 예측 실행")
print("=" * 80)

# 인덱스 계산
train_end_idx = len(train_data) - 1
valid_start_idx = train_end_idx + 1
valid_end_idx = train_end_idx + len(valid_data)
test_start_idx = valid_end_idx + 1
test_end_idx = valid_end_idx + len(test_data)

print(f"Train: idx 0 ~ {train_end_idx} ({train_end_idx + 1}개월)")
print(f"Valid: idx {valid_start_idx} ~ {valid_end_idx} ({len(valid_data)}개월)")
print(f"Test:  idx {test_start_idx} ~ {test_end_idx} ({len(test_data)}개월)")

# 결과 저장용
valid_predictions = {}
test_predictions = {}
valid_metrics = {}
test_metrics = {}

# 전체 실행 시작
section9_start = time.time()

for model_name in models_dict.keys():
    print(f"\n{'=' * 80}")
    print(f"[{model_name}] 예측 시작")
    print(f"{'=' * 80}")
    
    model = models_dict[model_name]
    
    # Valid 예측
    print(f"\n[{model_name}] Valid 기간 예측 중... (12개월)")
    valid_preds = recursive_forecast_extended(
        model=model,
        model_name=model_name,
        df_data=df_featured,
        scaler=scaler,
        imputer=imputer,
        feature_cols=feature_columns,
        start_idx=valid_start_idx,
        end_idx=valid_end_idx,
        target_col='승차인원수'
    )
    valid_predictions[model_name] = valid_preds
    
    # Valid 성능 평가
    valid_actual = valid_data['승차인원수'].values
    valid_rmse = np.sqrt(mean_squared_error(valid_actual, valid_preds))
    valid_mae = mean_absolute_error(valid_actual, valid_preds)
    valid_mape = np.mean(np.abs((valid_actual - valid_preds) / valid_actual)) * 100
    
    valid_metrics[model_name] = {
        'RMSE': valid_rmse,
        'MAE': valid_mae,
        'MAPE': valid_mape
    }
    
    print(f"✅ [{model_name}] Valid 완료")
    print(f"   - RMSE: {valid_rmse:,.0f}")
    print(f"   - MAE:  {valid_mae:,.0f}")
    print(f"   - MAPE: {valid_mape:.2f}%")
    
    # Test 예측 (Valid 예측값 반영)
    df_for_test = df_featured.copy()
    df_for_test.loc[valid_start_idx:valid_end_idx, '승차인원수'] = valid_preds
    
    print(f"\n[{model_name}] Test 기간 예측 중... (21개월)")
    test_preds = recursive_forecast_extended(
        model=model,
        model_name=model_name,
        df_data=df_for_test,
        scaler=scaler,
        imputer=imputer,
        feature_cols=feature_columns,
        start_idx=test_start_idx,
        end_idx=test_end_idx,
        target_col='승차인원수'
    )
    test_predictions[model_name] = test_preds
    
    # Test 통계
    test_mean = np.mean(test_preds)
    test_std = np.std(test_preds)
    test_min = np.min(test_preds)
    test_max = np.max(test_preds)
    
    test_metrics[model_name] = {
        'Mean': test_mean,
        'Std': test_std,
        'Min': test_min,
        'Max': test_max
    }
    
    print(f"✅ [{model_name}] Test 완료")
    print(f"   - 평균: {test_mean:,.0f}")
    print(f"   - 표준편차: {test_std:,.0f}")
    print(f"   - 범위: {test_min:,.0f} ~ {test_max:,.0f}")

section9_time = time.time() - section9_start
print(f"\n" + "=" * 80)
print(f"✅ 전체 8개 모델 Valid+Test 예측 완료 (총 소요: {section9_time/60:.1f}분)")
print(f"=" * 80)


📈 Section 9: Valid + Test 예측 실행
Train: idx 0 ~ 98 (99개월)
Valid: idx 99 ~ 110 (12개월)
Test:  idx 111 ~ 132 (22개월)

[RandomForest] 예측 시작

[RandomForest] Valid 기간 예측 중... (12개월)


[RandomForest] Recursive Forecast:   0%|          | 0/12 [00:00<?, ?it/s]

[RandomForest] Recursive Forecast: 100%|██████████| 12/12 [00:00<00:00, 33.09it/s]


✅ [RandomForest] Valid 완료
   - RMSE: 1,183,135
   - MAE:  1,171,194
   - MAPE: 67.33%

[RandomForest] Test 기간 예측 중... (21개월)


[RandomForest] Recursive Forecast: 100%|██████████| 22/22 [00:00<00:00, 33.46it/s]


...

✅ [LSTM] Test 완료
   - 평균: 18
   - 표준편차: 0
   - 범위: 18 ~ 18

[GRU] 예측 시작

[GRU] Valid 기간 예측 중... (12개월)


[GRU] Recursive Forecast: 100%|██████████| 12/12 [00:00<00:00, 205.58it/s]


✅ [GRU] Valid 완료
   - RMSE: 1,739,451
   - MAE:  1,731,609
   - MAPE: 100.00%

[GRU] Test 기간 예측 중... (21개월)


[GRU] Recursive Forecast: 100%|██████████| 22/22 [00:00<00:00, 191.63it/s]

✅ [GRU] Test 완료
   - 평균: 24
   - 표준편차: 1
   - 범위: 24 ~ 26

✅ 전체 8개 모델 Valid+Test 예측 완료 (총 소요: 0.0분)


In [53]:
# Section 10: Train+Valid+Test 통합 시각화
print("\n" + "=" * 80)
print("📊 Section 10: Train+Valid+Test 통합 시각화")
print("=" * 80)

fig, axes = plt.subplots(4, 2, figsize=(20, 20))
axes = axes.flatten()

model_names = list(models_dict.keys())

for idx, model_name in enumerate(model_names):
    ax = axes[idx]
    
    # Train 실제값
    ax.plot(train_data['Date'], train_data['승차인원수'], 
            label='Train (Actual)', color='blue', linewidth=1.5)
    
    # Valid 실제값 및 예측값
    ax.plot(valid_data['Date'], valid_data['승차인원수'], 
            label='Valid (Actual)', color='green', linewidth=1.5)
    ax.plot(valid_data['Date'], valid_predictions[model_name], 
            label='Valid (Predicted)', color='green', linestyle='--', linewidth=1.5, alpha=0.7)
    
    # Test 예측값
    ax.plot(test_data['Date'], test_predictions[model_name], 
            label='Test (Predicted)', color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    
    # 구분선
    ax.axvline(x=pd.to_datetime('2023-04-01'), color='gray', linestyle=':', alpha=0.5, label='Train/Valid Split')
    ax.axvline(x=pd.to_datetime('2024-04-01'), color='orange', linestyle=':', alpha=0.5, label='Valid/Test Split')
    
    # 제목 및 레이블
    valid_rmse = valid_metrics[model_name]['RMSE']
    valid_mape = valid_metrics[model_name]['MAPE']
    ax.set_title(f'{model_name} - Valid RMSE: {valid_rmse:,.0f}, MAPE: {valid_mape:.2f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.set_ylabel('승차인원수', fontsize=10)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
output_path = PROCESSED_DATA_PATH / 'full_period_forecast_8models.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"✅ 통합 시각화 저장: {output_path}")
plt.close()


📊 Section 10: Train+Valid+Test 통합 시각화
✅ 통합 시각화 저장: /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/processed_data/full_period_forecast_8models.png


In [54]:
# Section 11: Valid vs Test 성능 비교 및 월별 예측 테이블
print("\n" + "=" * 80)
print("📋 Section 11: Valid vs Test 성능 비교 및 월별 예측 테이블")
print("=" * 80)

# 성능 비교 DataFrame
comparison_data = []
for model_name in model_names:
    comparison_data.append({
        'Model': model_name,
        'Valid_RMSE': valid_metrics[model_name]['RMSE'],
        'Valid_MAE': valid_metrics[model_name]['MAE'],
        'Valid_MAPE': valid_metrics[model_name]['MAPE'],
        'Test_Mean': test_metrics[model_name]['Mean'],
        'Test_Std': test_metrics[model_name]['Std'],
        'Test_Min': test_metrics[model_name]['Min'],
        'Test_Max': test_metrics[model_name]['Max']
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n📊 Valid vs Test 성능 비교:")
print(df_comparison.to_string(index=False))

# 시각화: Valid RMSE 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Valid RMSE
ax1 = axes[0]
sorted_df = df_comparison.sort_values('Valid_RMSE')
ax1.barh(sorted_df['Model'], sorted_df['Valid_RMSE'], color='steelblue')
ax1.set_xlabel('Valid RMSE', fontsize=12)
ax1.set_title('Valid 성능 비교 (RMSE)', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
for i, (model, rmse) in enumerate(zip(sorted_df['Model'], sorted_df['Valid_RMSE'])):
    ax1.text(rmse, i, f' {rmse:,.0f}', va='center', fontsize=10)

# Right: Test 예측 평균
ax2 = axes[1]
ax2.barh(df_comparison['Model'], df_comparison['Test_Mean'], color='coral')
ax2.set_xlabel('Test 예측 평균', fontsize=12)
ax2.set_title('Test 예측 평균 비교', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
for i, (model, mean_val) in enumerate(zip(df_comparison['Model'], df_comparison['Test_Mean'])):
    ax2.text(mean_val, i, f' {mean_val:,.0f}', va='center', fontsize=10)

plt.tight_layout()
output_path2 = PROCESSED_DATA_PATH / 'valid_vs_test_comparison.png'
plt.savefig(output_path2, dpi=150, bbox_inches='tight')
print(f"\n✅ 성능 비교 시각화 저장: {output_path2}")
plt.close()

# 월별 예측 테이블 생성
monthly_data = {
    'Date': test_data['Date'].values
}
for model_name in model_names:
    monthly_data[model_name] = test_predictions[model_name]

df_monthly = pd.DataFrame(monthly_data)
df_monthly['Mean'] = df_monthly[model_names].mean(axis=1)
df_monthly['Std'] = df_monthly[model_names].std(axis=1)
df_monthly['Min'] = df_monthly[model_names].min(axis=1)
df_monthly['Max'] = df_monthly[model_names].max(axis=1)

# CSV 저장
csv_path = PROCESSED_DATA_PATH / 'monthly_forecast_8models.csv'
df_monthly.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✅ 월별 예측 테이블 저장: {csv_path} ({len(df_monthly)}행)")

# 월별 예측 범위 시각화
fig, ax = plt.subplots(figsize=(14, 6))

# Fill area (Min ~ Max)
ax.fill_between(df_monthly['Date'], df_monthly['Min'], df_monthly['Max'], 
                alpha=0.3, color='lightblue', label='예측 범위 (Min-Max)')

# 평균선
ax.plot(df_monthly['Date'], df_monthly['Mean'], 
        color='red', linewidth=2, label='평균 예측', marker='o')

# 8개 모델 얇은 선
for model_name in model_names:
    ax.plot(df_monthly['Date'], df_monthly[model_name], 
            linewidth=0.8, alpha=0.5, label=model_name)

ax.set_title('월별 예측 범위 (8개 모델)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('승차인원수', fontsize=12)
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
output_path3 = PROCESSED_DATA_PATH / 'monthly_forecast_range.png'
plt.savefig(output_path3, dpi=150, bbox_inches='tight')
print(f"✅ 월별 예측 범위 시각화 저장: {output_path3}")
plt.close()


📋 Section 11: Valid vs Test 성능 비교 및 월별 예측 테이블

📊 Valid vs Test 성능 비교:
       Model   Valid_RMSE    Valid_MAE  Valid_MAPE    Test_Mean     Test_Std     Test_Min     Test_Max
RandomForest 1.183135e+06 1.171194e+06   67.334716 5.505364e+05  2949.784106 5.427794e+05 5.536511e+05
     XGBoost 1.377717e+06 1.367802e+06   78.803370 3.638188e+05     0.000000 3.638188e+05 3.638188e+05
    LightGBM 9.132023e+05 8.993069e+05   51.554725 7.995792e+05 14951.051224 7.739960e+05 8.249474e+05
    CatBoost 6.526217e+05 6.318883e+05   35.941139 1.054801e+06  8202.276288 1.040611e+06 1.067563e+06
         MLP 1.736457e+06 1.728591e+06   99.823254 3.272468e+02     0.527906 3.265152e+02 3.284158e+02
         RNN 1.739455e+06 1.731612e+06   99.999504 1.696092e+01     0.010444 1.695377e+01 1.699903e+01
        LSTM 1.739455e+06 1.731612e+06   99.999508 1.831789e+01     0.000012 1.831787e+01 1.831791e+01
         GRU 1.739451e+06 1.731609e+06   99.999307 2.437657e+01     0.675389 2.353124e+01 2.551014e+01

✅

In [55]:
# Section 12: 최종 결과 저장 및 요약
print("\n" + "=" * 80)
print("💾 Section 12: 최종 결과 저장 및 요약")
print("=" * 80)

# Best Model 선택 (Valid RMSE 기준)
best_model_name = min(valid_metrics, key=lambda k: valid_metrics[k]['RMSE'])
print(f"\n🏆 Best Model (Valid RMSE 기준): {best_model_name}")
print(f"   - Valid RMSE: {valid_metrics[best_model_name]['RMSE']:,.0f}")
print(f"   - Valid MAPE: {valid_metrics[best_model_name]['MAPE']:.2f}%")

# Best Model의 전체 예측 결과 저장
final_results = pd.DataFrame({
    'Date': pd.concat([train_data['Date'], valid_data['Date'], test_data['Date']]).reset_index(drop=True),
    'Type': ['Train'] * len(train_data) + ['Valid'] * len(valid_data) + ['Test'] * len(test_data),
    'Actual': list(train_data['승차인원수']) + list(valid_data['승차인원수']) + [np.nan] * len(test_data),
    'Predicted': list(train_data['승차인원수']) + list(valid_predictions[best_model_name]) + list(test_predictions[best_model_name])
})

final_csv_path = RESULT_PATH / f'final_forecast_{best_model_name}.csv'
final_results.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
print(f"\n✅ Best Model 예측 결과 저장: {final_csv_path}")

# 전체 모델 성능 요약
print("\n" + "=" * 80)
print("📊 전체 모델 성능 요약 (Valid 기준)")
print("=" * 80)
sorted_models = sorted(valid_metrics.items(), key=lambda x: x[1]['RMSE'])
for rank, (model_name, metrics) in enumerate(sorted_models, 1):
    print(f"{rank}. {model_name:12s} - RMSE: {metrics['RMSE']:>10,.0f}  |  MAE: {metrics['MAE']:>10,.0f}  |  MAPE: {metrics['MAPE']:>6.2f}%")

# 실행 시간 요약
total_time = time.time() - start_time
print("\n" + "=" * 80)
print("⏱️  실행 시간 요약")
print("=" * 80)
print(f"총 소요 시간: {total_time/60:.1f}분 ({total_time:.0f}초)")

# 생성된 파일 목록
print("\n" + "=" * 80)
print("📁 생성된 파일 목록")
print("=" * 80)
print(f"1. {PROCESSED_DATA_PATH / 'full_period_forecast_8models.png'}")
print(f"2. {PROCESSED_DATA_PATH / 'valid_vs_test_comparison.png'}")
print(f"3. {PROCESSED_DATA_PATH / 'monthly_forecast_range.png'}")
print(f"4. {PROCESSED_DATA_PATH / 'monthly_forecast_8models.csv'}")
print(f"5. {final_csv_path}")

print("\n" + "=" * 80)
print("✅ 전체 작업 완료!")
print("=" * 80)
print(f"⏱️  종료 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}")


💾 Section 12: 최종 결과 저장 및 요약

🏆 Best Model (Valid RMSE 기준): CatBoost
   - Valid RMSE: 652,622
   - Valid MAPE: 35.94%

✅ Best Model 예측 결과 저장: /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/result/final_forecast_CatBoost.csv

📊 전체 모델 성능 요약 (Valid 기준)
1. CatBoost     - RMSE:    652,622  |  MAE:    631,888  |  MAPE:  35.94%
2. LightGBM     - RMSE:    913,202  |  MAE:    899,307  |  MAPE:  51.55%
3. RandomForest - RMSE:  1,183,135  |  MAE:  1,171,194  |  MAPE:  67.33%
4. XGBoost      - RMSE:  1,377,717  |  MAE:  1,367,802  |  MAPE:  78.80%
5. MLP          - RMSE:  1,736,457  |  MAE:  1,728,591  |  MAPE:  99.82%
6. GRU          - RMSE:  1,739,451  |  MAE:  1,731,609  |  MAPE: 100.00%
7. RNN          - RMSE:  1,739,455  |  MAE:  1,731,612  |  MAPE: 100.00%
8. LSTM         - RMSE:  1,739,455  |  MAE:  1,731,612  |  MAPE: 100.00%

⏱️  실행 시간 요약
총 소요 시간: 0.4분 (26초)

📁 생성된 파일 목록
1. /mnt/c/Users/Admin/PycharmProjects/Demand-Forecasting-Dive/processed_data/full_period_forecast_8models.p